<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 23 · Model Risk, Calibration Risk, and Verification

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch23_model_risk_calibration_risk_verification.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Fixed-income analytics fail quietly: a wrong day count, a flipped sign, or a mis-specified curve can still print plausible numbers. Verification is therefore part of the modeling routine, not a final cosmetic step. You will reproduce three checks: a price/yield round trip on a semiannual coupon bond, a par-swap repricing residual that should be zero to ten decimal places, and a finite-difference DV01 compared with a duration approximation. You will also read calibration residuals in basis points for a linear and a smooth curve fit, and identify specification error as a failure mode distinct from noisy data.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script prices a 4.5 percent semiannual bond from a frozen cash-flow table, recovers its yield by bisection, and reprices a par swap on its own discount factors. It then prints a finite-difference DV01 of 0.018921 against a duration approximation of 0.018924, and calibration residuals in basis points where the linear fit misses by as much as 7 bp while the smooth fit stays within 2 bp.

In [ ]:
SCRIPT = ROOT / "code" / "ch23_model_risk_calibration_risk_verification.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Match every printed check to the failure mode it targets.

- The price/yield round trip prices the bond at exactly 100.0 and recovers 0.045 with a printed error of -0.0; the recovered yield must reprice the input price, not merely look plausible.
- The swap repricing residual is 0.0000000000: the par rate equates the fixed and floating legs on the same discount factors.
- Finite-difference DV01 0.018921 and duration-based DV01 0.018924 agree to within 3e-6 currency units per basis point, validating sign and scale together.
- Calibration residuals are stated in basis points: the linear fit ranges from -7 to +6 bp, while the smooth fit stays within 2 bp of every quote.
- Yields are decimals with semiannual compounding (frequency 2), and payment times are in years.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch23_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Calibration error from a mis-specified curve fit

A true yield curve is generated as a quadratic in maturity, and 2 bp of seeded noise is added to produce twelve synthetic quotes in decimal rate units. A straight line is fitted to those quotes by least squares, then the correctly specified quadratic is fitted to the same data. Both fits use identical inputs; only the model differs. The comparison isolates specification risk: the mis-specified fit's residuals are driven by curvature the line cannot represent, not by the noise. Residuals are reported in basis points and maturities in years.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)  # fixed seed for the quote noise
maturity = np.linspace(0.5, 10.0, 12)  # quoted maturities in years
true = 0.02 + 0.002 * maturity + 0.00008 * maturity ** 2
quotes = true + rng.normal(0.0, 0.0002, 12)  # 2 bp of quote noise

def residuals(degree):
    design = np.vander(maturity, degree + 1)  # basis functions
    coef = np.linalg.lstsq(design, quotes, rcond=None)[0]
    return (design @ coef - quotes) * 1e4  # residuals in bp

for degree in (1, 2):
    resid = residuals(degree)
    rmse = float(np.sqrt(np.mean(resid ** 2)))
    print(f"degree {degree}: rmse {rmse:0.2f} bp, "
          f"max {np.abs(resid).max():0.2f} bp")

### What to observe

The straight-line fit has an RMSE of 6.76 bp with residuals up to 11.59 bp, while the correctly specified quadratic fits the same quotes with an RMSE of 1.18 bp, close to the 2 bp noise level. Specification error, not data quality, drives the mis-specified fit's residuals.

## Exercises

Each check targets a known failure mode; name the one you are testing as you complete it.

1. **Round-trip the yield.** Price the bond at 4.5 percent, confirm the price of 100.0, and recover the yield from that price. *Hint:* Use semiannual discounting with frequency 2.
2. **Stress the bracket.** Attempt to invert a price below the 20 percent yield floor and describe what a robust function should do. *Hint:* The script rejects prices outside its zero-to-20 percent bracket instead of returning an endpoint.
3. **Verify the swap identity.** Recompute the par rate and show the fixed leg minus float leg residual is zero to ten decimal places. *Hint:* The floating leg equals one minus the final discount factor.
4. **Quantify DV01 error.** Compare the finite-difference DV01 of 0.018921 with the duration approximation of 0.018924 and explain the small gap. *Hint:* The approximation ignores convexity and higher-order terms.
5. **Read the residual pattern.** Explain why the linear fit alternates residuals up to 7 bp while the smooth fit stays within 2 bp. *Hint:* Compare each model's shape with the quoted term structure.

For more detail, see Chapter 23 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">